# nathwaniGPT — LoRA Fine-Tune for Coding (Colab)

**Runtime:** Runtime → Change runtime type → T4 GPU (free tier)

Run each cell in order. When you hit **Step 4**, you'll be prompted to upload your dataset file.

**Supported dataset formats:**
- `.jsonl` with `instruction` + `output` fields
- `.jsonl` with `prompt` + `completion` fields
- `.csv` with the same field names

Example row:
```json
{"instruction": "Write a Python function that reverses a linked list.", "output": "def reverse_linked_list(head):\n    ..."}
```

## Step 1 — Install Dependencies
Takes ~3 minutes on a fresh runtime.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
print('Done.')

## Step 2 — Load Base Model
Loads qwen2.5:14b at 4-bit quantization (~9GB VRAM). If you get an OOM error, switch to `unsloth/Qwen2.5-7B-bnb-4bit` in the line below.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-14B-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print('Model loaded.')

## Step 3 — Attach LoRA Adapters
`r=16` is a good balance of quality vs training time. Increase to 32 for more expressive adapters (slower).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('LoRA adapters attached.')
model.print_trainable_parameters()

## Step 4 — Upload Your Dataset
A file picker will appear. Upload your `.jsonl` or `.csv` file.

In [ ]:
from google.colab import files
import json, io
import pandas as pd

uploaded = files.upload()
filename = list(uploaded.keys())[0]
raw = uploaded[filename].decode('utf-8')

if filename.endswith('.jsonl'):
    data = [json.loads(line) for line in raw.splitlines() if line.strip()]
elif filename.endswith('.csv'):
    data = pd.read_csv(io.StringIO(raw)).to_dict('records')
else:
    raise ValueError('Unsupported format. Use .jsonl or .csv')

print(f'Loaded {len(data)} examples')
print('Sample:', data[0])

## Step 5 — Format Dataset
Wraps each example in nathwaniGPT's system prompt and chat template.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are nathwaniGPT, a sharp and highly capable AI assistant specialised in coding. "
    "You think carefully before responding — when a problem is complex, you reason through it "
    "step by step before giving your answer. When the answer is simple, you give it directly "
    "without theatrics. You write clean, idiomatic code with brief explanations where useful. "
    "You never pad responses, add unnecessary caveats, or repeat yourself. "
    "When you are uncertain, you say so plainly. You treat the user as an intelligent adult."
)

def get_instruction(row):
    # support both field naming conventions
    return row.get('instruction') or row.get('prompt', '')

def get_output(row):
    return row.get('output') or row.get('completion', '')

def format_row(row):
    instruction = get_instruction(row)
    output = get_output(row)
    input_ctx = row.get('input', '').strip()
    if input_ctx:
        instruction = f"{instruction}\n\n{input_ctx}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

formatted = [format_row(r) for r in data]
dataset = Dataset.from_list(formatted)
print(f'Dataset ready. Example:')
print(dataset[0]['text'][:500])

## Step 6 — Train
Adjust `num_train_epochs` if you want more or fewer passes over the data. 3 is a safe default.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/tmp/nathwaniGPT-checkpoints",
        report_to="none",
    ),
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'Done. Runtime: {trainer_stats.metrics["train_runtime"]:.0f}s')

## Step 7 — Export to GGUF & Download
Exports as Q4_K_M GGUF (same format as your local Ollama model). File will download automatically — it'll be ~9GB so give it a few minutes.

In [ ]:
print('Saving LoRA adapter...')
model.save_pretrained('/tmp/nathwaniGPT-lora')
tokenizer.save_pretrained('/tmp/nathwaniGPT-lora')

print('Converting to GGUF (Q4_K_M)...')
model.save_pretrained_gguf('/tmp/nathwaniGPT-gguf', tokenizer, quantization_method='q4_k_m')

print('Downloading...')
files.download('/tmp/nathwaniGPT-gguf/model-q4_k_m.gguf')
print('Done. See the instructions below for loading into Ollama.')

## Step 8 — Load into Ollama (run this on your Mac)

Once the GGUF file has downloaded, move it into your nathwaniGPT repo and run:

```bash
# from inside ~/nathwaniGPT
mkdir -p models/v1.8-beta
mv ~/Downloads/model-q4_k_m.gguf models/v1.8-beta/nathwaniGPT-v1.8-beta.gguf
```

Then create a Modelfile at `models/v1.8-beta/Modelfile`:

```
FROM ./nathwaniGPT-v1.8-beta.gguf

PARAMETER num_ctx 8192
PARAMETER temperature 0.5
PARAMETER top_p 0.85
PARAMETER top_k 40
PARAMETER repeat_penalty 1.05
PARAMETER num_predict -1
```

Then:

```bash
ollama create nathwaniGPT:v1.8-beta -f models/v1.8-beta/Modelfile
ollama run nathwaniGPT:v1.8-beta
```
